# 85 · Sharp or gradual: how segment-level pathways change along the elastic curve

A segment-resolution study calls pathways "enriched in PTS1, PTS2 or PTS3". At that resolution a step at a segment
boundary and a gradient across segments look alike. This notebook measures, along the package's elastic curve, the
width of each pathway's change (sharp ≤ 0.5 segment units, gradual ≥ 1), where it happens, and whether the curve
resolves that difference at that place.

**Protocol.** `docs/results/pt-transition-sharpness.md` (frozen in 1971e97; amended before any fit in 72ad1fb).

| Step | What |
|---|---|
| S0 | Mouse units (the mice's own label transitions); earlier pooled-unit positions re-read |
| P | Equal-count bins (40 per mouse), log2 rate per bin, centred by the gene mean and the median usable gene |
| E, R | One- or two-edge logistic fits; width = 10–90% distance; edges agreed between the two mice |
| Q | What the curve can resolve: spline and binning step responses, label steps, sharpest agreed gene edges |
| G | Gene edges and the composition of each pathway edge |

**What bounds the reading.** C14 (mouse fold noise ÷ within-segment spread 4.2, 2.0, 0.98 in S1–S3); two male
control mice, edges agreed between them, not tested; every class is descriptive. Human is not analysed: its label
transitions cross on the curve (issue 6).

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')                       # the fits run in parallel processes; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'                  # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only: universe, sets, stage cache
T72 = results_root / 'pt_package_coordinate_validation'           # read only: transitions
T84 = results_root / 'pt_within_segment_pathways'                 # read only: Chen-called pathways and cores
CHEN = data_root / 'external' / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz'
output = results_root / 'pt_transition_sharpness'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DS = 'human_vs_mouse'
CHEN_SHA = 'daf8b6f46bf772bbbc950b68d688b7540528e2ec52b357440fac1ebe224d5488'
REQUIRED = [EXPORT, CHEN, upstream13 / 'ortholog_map_used.csv', T72 / 'tables' / 'transitions.csv',
            T84 / 'tables' / 'segment_pathways.csv', T84 / 'run_record.json',
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72 and 84 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '85.transition_sharpness.1'   # Bump when a calculation changes (no stage cache here).
FINGERPRINT = 'efafc9ac2e99'                           # confirmed hvm pass-1 labels (notebook 70)
SEGMENTS = ('S1', 'S2', 'S3')
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
MOUSE = NAMES[:2]
MIN_DETECTION, MIN_CORE = .05, 5                       # 81's usable rule; 84's core rule
N_BINS = 40                                            # P1
FLAT, SHARP, GRADUAL, WINDOW, AGREE = .25, .5, 1., .15, .25   # E3, E4, E5, R1
STEP_AT = (.5, 1., 1.5, 2., 2.5)                       # Q1, Q2
Q4_PERCENTILE, Q4_MIN = 5, 20                          # Q4
NEAR = .5                                              # G2
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9'}
SEG_COLOR = {'S1': '#E69F00', 'S2': '#009E73', 'S3': '#CC79A7'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def short(pathway):
    lib, name = pathway.split('::', 1)
    name = re.sub(r'\s+R-HSA-\d+$', '', name)
    return f"{ {'Reactome_2022': 'R', 'KEGG_2019_Mouse': 'K', 'MSigDB_Hallmark_2020': 'H'}[lib] }: {name}"


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

G1 the Chen sha256; G2 `hvm_inputs` reproduces 71's universe, sets and units, and 71's cached condition model its
T_spatial; G3 the logistic transitions reproduce 72's; G4 84's pathways and cores, every core gene usable in both mice.

In [ ]:
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import common_grid
from pseudospace import published_findings as pf
from pseudospace import species_calls as sc
from pseudospace import transition_sharpness as ts

t0 = time.perf_counter()
assert hashlib.sha256(CHEN.read_bytes()).hexdigest() == CHEN_SHA, 'G1: the Chen file differs from its sha256'
GUARDS.append('G1: Chen 2021 TPM file matches its sha256')

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'hvm')
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
GIDX = {g: i for i, g in enumerate(GENES)}
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G2: universe differs from 71'
assert INP['gene_sets'] == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G2: gene sets differ from 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = np.asarray(INP['segment']).astype(str)
LIBRARY = np.asarray(INP['library'], float)
COUNTS = INP['counts'].tocsr()
STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
COND = stored(T71, f'gene_model_{DS}_species')
np.testing.assert_allclose(COND['stat|T_spatial'], STAT71['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
GRID = COND['grid']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
KNOTS = COND['knots']
GUARDS.append(f'G2: {len(GENES):,} genes and {len(POSITION):,} units equal 71; 71\'s cached condition model reproduces '
              f'its T_spatial; spline knots {np.round(KNOTS, 3).tolist()}')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, SEGMENT, SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)
GUARDS.append('G3: logistic transitions reproduce 72\'s')

DETECTION = {spec: np.asarray((COUNTS[SPECIMEN == spec] > 0).mean(axis=0)).ravel() for spec in MOUSE}
USABLE = {spec: (COND[f'converged|{spec}'].astype(bool) & ~COND[f'separated|{spec}'].astype(bool)
                 & (DETECTION[spec] >= MIN_DETECTION)) for spec in MOUSE}
BOTH = USABLE[MOUSE[0]] & USABLE[MOUSE[1]]

S84 = pd.read_csv(T84 / 'tables' / 'segment_pathways.csv')
assert json.loads((T84 / 'run_record.json').read_text())['notebook_logic_version'] == '84.within_segment_pathways.1'
S84 = S84[S84.core >= MIN_CORE].reset_index(drop=True)
assert not S84.pathway.str.contains('|', regex=False).any()
S84['key'] = S84.segment + '|' + S84.pathway                   # one pathway can be called in two segments
CORES = {r.key: r['core genes'].split(', ') for _, r in S84.iterrows()}
REPS = S84.loc[S84.representative.astype(bool), 'key'].tolist()
assert S84.groupby('segment').size().to_dict() == {'S1': 9, 'S2': 53, 'S3': 1}, S84.groupby('segment').size()
assert all(len(c) == n for c, n in zip(CORES.values(), S84.core)), 'G4: core lists differ from their counts'
missing = sorted({g for c in CORES.values() for g in c if g not in GIDX or not BOTH[GIDX[g]]})
assert not missing, f'G4: core genes not usable in both mice: {missing[:10]}'
GUARDS.append(f'G4: 84\'s {len(S84)} pathways with ≥ 5 core genes (S1 9, S2 53, S3 1; {len(REPS)} representatives); '
              'every core gene usable in both mice')
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 3 · S0 · Mouse units, and earlier positions re-read

Mouse units: the mean of the two mice's own logistic label transitions. The pooled units of 79–84 used all four
specimens. The table converts the mouse positions quoted by 81, 83 and 84 (arithmetic, not a fit).

In [ ]:
POOLED = sc.mean_transitions(TRANSITIONS, NAMES)
MOUSE_CUTS = sc.mean_transitions(TRANSITIONS, MOUSE)
print(f'pooled cuts {POOLED[0]:.3f}, {POOLED[1]:.3f} · mouse cuts {MOUSE_CUTS[0]:.3f}, {MOUSE_CUTS[1]:.3f}')


def to_mouse(position):
    return sc.to_segment_units(position, MOUSE_CUTS)


def from_units(units, cuts):
    """Inverse of the piecewise-linear segment units (lo 0, hi 1)."""
    u = np.asarray(units, float)
    t12, t23 = cuts
    return np.where(u < 1, u * t12, np.where(u < 2, t12 + (u - 1) * (t23 - t12), t23 + (u - 2) * (1 - t23)))


QUOTED = [('83', 'NaPi-IIa (Slc34a1) switch', 2.37), ('83', 'NaPi-IIc (Slc34a3) switch', .69),
          ('83', 'S-M1 (CellSpectra mouse-high S1) fall', 2.13), ('81', 'S-M1 fall, half done', 2.18),
          ('83', 'U-M3 (CellSpectra mouse-high S3-only) rise', .72), ('81', 'U-M3 rise, half done', .80),
          ('84', 'pooled S1 start', 0.), ('84', 'pooled S1/S2 boundary', 1.), ('84', 'pooled S2/S3 boundary', 2.)]
S0 = pd.DataFrame([{'notebook': nb, 'what': what, 'pooled units': u, 'curve position': float(from_units(u, POOLED)),
                    'mouse units': float(to_mouse(from_units(u, POOLED))),
                    'mouse reading': pf.segment_position(float(to_mouse(from_units(u, POOLED))))}
                   for nb, what, u in QUOTED])
display(S0.round(3))

## 4 · P · Binned profiles per mouse (no spline)

In [ ]:
BINS, X, PROFILE, REF = {}, {}, {}, {}
for spec in MOUSE:
    rows = np.flatnonzero(SPECIMEN == spec)
    b = ts.equal_count_bins(POSITION[rows], N_BINS)
    BINS[spec] = (rows, b)
    X[spec] = to_mouse(np.array([np.median(POSITION[rows][b == k]) for k in range(N_BINS)]))
    raw = ts.binned_log_rates(COUNTS[rows], LIBRARY[rows], b, N_BINS)
    REF[spec] = pf.reference_curve(raw, USABLE[spec])
    PROFILE[spec] = raw - raw.mean(axis=1, keepdims=True) - REF[spec]
    print(f'{spec}: {len(rows):,} units, {np.bincount(b).min()}–{np.bincount(b).max()} per bin; '
          f'bins span {X[spec][0]:.2f}–{X[spec][-1]:.2f} mouse units; median spacing {np.median(np.diff(X[spec])):.3f}')


def pathway_profile(pathway, spec):
    idx = [GIDX[g] for g in CORES[pathway]]
    return np.median(PROFILE[spec][idx], axis=0)


PATHWAYS = list(CORES)
PW = {spec: np.array([pathway_profile(p, spec) for p in PATHWAYS]) for spec in MOUSE}

## 5 · Q1–Q3 · What a true step looks like on this curve

In [ ]:
q_rows = []
for spec in MOUSE:
    rows, b = BINS[spec]
    pos = POSITION[rows]
    grid_units = to_mouse(GRID)
    for x0 in STEP_AT:
        raw0 = float(from_units(x0, MOUSE_CUTS))
        curve = ts.spline_step_response(pos, raw0, KNOTS, GRID)
        frac = np.bincount(b, weights=(pos >= raw0).astype(float), minlength=N_BINS) / np.bincount(b, minlength=N_BINS)
        _, e = ts.fit_edges(X[spec], frac[None], flat=.1)
        q_rows.append({'reference': 'Q1 spline step response', 'specimen': spec, 'at': x0,
                       'width': ts.crossing_width(grid_units, curve)})
        q_rows.append({'reference': 'Q2 binning step response', 'specimen': spec, 'at': x0,
                       'width': float(e.loc[(e.position - x0).abs().idxmin(), 'width']) if len(e) else np.nan})
LABEL_FRAC = {}
for spec in MOUSE:
    rows, b = BINS[spec]
    seg = SEGMENT[rows]
    n = np.bincount(b, minlength=N_BINS)
    for boundary, later in ((1, ('S2', 'S3')), (2, ('S3',))):
        is_later = np.array([any(s.endswith(l) for l in later) for s in seg], float)
        frac = np.bincount(b, weights=is_later, minlength=N_BINS) / n
        LABEL_FRAC[(spec, boundary)] = frac
        _, e = ts.fit_edges(X[spec], frac[None], flat=.1)
        e = e[e.direction > 0]
        q_rows.append({'reference': 'Q3 label step', 'specimen': spec, 'at': boundary,
                       'width': float(e.loc[(e.position - boundary).abs().idxmin(), 'width']) if len(e) else np.nan,
                       'fitted position': float(e.loc[(e.position - boundary).abs().idxmin(), 'position']) if len(e) else np.nan})
RESOLUTION = pd.DataFrame(q_rows)
display(RESOLUTION.pivot_table(index=['reference', 'at'], columns='specimen', values='width').round(3))
LABEL_WIDTH = RESOLUTION[RESOLUTION.reference.eq('Q3 label step')].groupby('at').width.mean().to_dict()

## 6 · G1 · Edges of every gene usable in both mice

The fits run in parallel chunks; each gene's agreed edges (R1–R2) are kept. Q4 then reads the sharpest agreed edges
inside each segment.

In [ ]:
from joblib import Parallel, delayed

GENE_IDX = np.flatnonzero(BOTH)


def fit_chunk(x, block):
    models, edges = ts.fit_edges(x, block)
    return models, edges


def fit_many(spec, rows):
    chunks = np.array_split(np.arange(len(rows)), max(1, N_JOBS * 4))
    out = Parallel(n_jobs=N_JOBS)(delayed(fit_chunk)(X[spec], PROFILE[spec][rows[c]]) for c in chunks if len(c))
    models, edges = [], []
    for c, (m, e) in zip([c for c in chunks if len(c)], out):
        m['profile'] = c[m.profile.to_numpy()]
        e['profile'] = c[e.profile.to_numpy()] if len(e) else e['profile']
        models.append(m)
        edges.append(e)
    return pd.concat(models, ignore_index=True), pd.concat(edges, ignore_index=True)


t1 = time.perf_counter()
GENE_FITS = {spec: fit_many(spec, GENE_IDX) for spec in MOUSE}
print(f'Gene fits: {len(GENE_IDX):,} genes x 2 mice in {time.perf_counter() - t1:.0f} s')
agreed = []
edges_by = {spec: dict(tuple(GENE_FITS[spec][1].groupby('profile'))) for spec in MOUSE}
empty = GENE_FITS[MOUSE[0]][1].iloc[:0]
for i in range(len(GENE_IDX)):
    a = edges_by[MOUSE[0]].get(i, empty).reset_index(drop=True)
    b = edges_by[MOUSE[1]].get(i, empty).reset_index(drop=True)
    if len(a) and len(b):
        ag = ts.agreed_edges(a, b, max_distance=AGREE, window=WINDOW)
        if len(ag):
            agreed.append(ag.assign(gene=GENES[GENE_IDX[i]]))
GENE_EDGES = pd.concat(agreed, ignore_index=True)
gene_summary = pd.DataFrame({spec: GENE_FITS[spec][0].model.value_counts() for spec in MOUSE}).fillna(0).astype(int)
display(gene_summary)
print(f'{GENE_EDGES.gene.nunique():,} genes with ≥ 1 agreed edge; {len(GENE_EDGES):,} agreed edges')
display(pd.crosstab(GENE_EDGES.location, GENE_EDGES['class']))

Q4 = {}
for k, seg in enumerate(SEGMENTS):
    inside = GENE_EDGES[GENE_EDGES.location.eq(f'inside {seg}')]
    ok = inside[inside['class'].ne('mice disagree on width')]
    Q4[seg] = {'agreed interior edges': len(ok),
               'reference width': float(np.percentile(ok.width, Q4_PERCENTILE)) if len(ok) >= Q4_MIN else np.nan,
               'sharp agreed interior edges': int((ok['class'] == 'sharp').sum())}
Q4 = pd.DataFrame(Q4).T
display(Q4.round(3))
for seg, r in Q4.iterrows():
    RESOLUTION = pd.concat([RESOLUTION, pd.DataFrame([{'reference': 'Q4 interior (5th percentile of agreed gene edges)',
                                                       'specimen': 'both', 'at': seg, 'width': r['reference width'],
                                                       'edges': r['agreed interior edges']}])], ignore_index=True)

## 7 · E, R · Pathway edges, agreed between the mice, with their class, place and resolution

In [ ]:
PW_FITS = {spec: ts.fit_edges(X[spec], PW[spec]) for spec in MOUSE}


def reference_at(location):
    if location == 'at S1/S2':
        return LABEL_WIDTH.get(1, np.nan)
    if location == 'at S2/S3':
        return LABEL_WIDTH.get(2, np.nan)
    return float(Q4.loc[location.split()[-1], 'reference width']) if location else np.nan


def reading(row):
    if row['class'] in ('sharp', 'mice disagree on width'):
        return row['class']
    ref = row['step reference']
    if np.isfinite(ref) and ref <= SHARP:
        return row['class']
    return f"{row['class']}, or a blurred step (unresolved here)"


rows = []
for i, p in enumerate(PATHWAYS):
    a = PW_FITS[MOUSE[0]][1].query('profile == @i').reset_index(drop=True)
    b = PW_FITS[MOUSE[1]][1].query('profile == @i').reset_index(drop=True)
    ag = ts.agreed_edges(a, b, max_distance=AGREE, window=WINDOW) if len(a) and len(b) else pd.DataFrame()
    base = {'segment (Chen)': p.split('|')[0], 'key': p, 'pathway': p.split('|', 1)[1], 'name': short(p.split('|', 1)[1]),
            'core': len(CORES[p]),
            'representative': p in REPS,
            'model A': PW_FITS[MOUSE[0]][0].loc[i, 'model'], 'model B': PW_FITS[MOUSE[1]][0].loc[i, 'model'],
            'edges A': len(a), 'edges B': len(b)}
    if not len(ag):
        flat = base['model A'] == base['model B'] == 'flat'
        rows.append({**base, 'edge': -1, 'main': True, 'class': 'flat' if flat else 'no agreed edge'})
        continue
    main = int(ag.amplitude.abs().idxmax())
    for j, e in ag.iterrows():
        row = {**base, 'edge': j, 'main': j == main, **e.to_dict()}
        row['step reference'] = reference_at(e['location'])
        row['reading'] = reading(row)
        row.update(ts.edge_composition(e, GENE_EDGES, CORES[p], near=NEAR))
        row['mouse position'] = pf.segment_position(e['position'])
        rows.append(row)
EDGES = pd.DataFrame(rows)
MAIN = EDGES[EDGES.main].copy()
cols = ['segment (Chen)', 'name', 'core', 'class', 'mouse position', 'location', 'width', 'width A', 'width B',
        'amplitude', 'step reference', 'reading', 'composition', 'genes', 'genes sharp', 'genes gradual', 'position IQR']
display(MAIN[MAIN.representative][cols].round(2))
summary = pd.crosstab([MAIN['segment (Chen)']], [MAIN['class']])
display(summary)
display(pd.crosstab([MAIN['segment (Chen)'], MAIN['location'].fillna('')], MAIN['reading'].fillna(MAIN['class'])))

## 7b · Post hoc checks (added after the first run; recorded as deviations)

- **PH1 · Chance agreement of gene edges.** Q4 came out at the fit's lower bound (0.05) in every segment. Gene i's
  mouse-A edges are paired with a random gene's mouse-B edges (shuffled within tertiles of mouse mean count; 5
  shuffles). Q4′ is the 5th percentile of agreed interior widths among edges with |amplitude| ≥ A*. A* is the smallest
  of 0.25, 0.5, 0.75, 1, 1.5 and 2 log2 at which the shuffled count is ≤ 10% of the observed count in that interior.
- **PH2 · Unit identity along the curve.** The fraction of each bin's units whose package `primary_identity` is
  Thin_Limb. An edge whose midpoint sits where that fraction (mean of the mice) is ≥ 0.25 is read "at the PT →
  thin-limb identity switch", not as a change inside the PT.
- **PH3 · The spline step response in pooled units**, the units in which 81 applied its thresholds.

In [ ]:
OBS = INP['obs']
IDENT = OBS['primary_identity'].astype(str).to_numpy()
THIN = {}
for spec in MOUSE:
    rows, b = BINS[spec]
    THIN[spec] = np.bincount(b, weights=(IDENT[rows] == 'Thin_Limb').astype(float), minlength=N_BINS) / np.bincount(b, minlength=N_BINS)


def thin_at(x):
    return float(np.mean([np.interp(x, X[s], THIN[s]) for s in MOUSE]))


identity = pd.DataFrame({f'x {s}': X[s] for s in MOUSE} | {f'thin-limb identity {s}': THIN[s] for s in MOUSE}
                        | {f'PT identity {s}': np.bincount(BINS[s][1], weights=(IDENT[BINS[s][0]] == 'Proximal_Tubule').astype(float),
                                                           minlength=N_BINS) / np.bincount(BINS[s][1], minlength=N_BINS) for s in MOUSE})
display(identity.iloc[24:].round(2))
S0['thin-limb identity there'] = [thin_at(u) for u in S0['mouse units']]
display(S0[['notebook', 'what', 'mouse reading', 'thin-limb identity there']].round(2))

rng = np.random.default_rng(85)
log_mean = np.log(np.asarray(COUNTS[np.isin(SPECIMEN, MOUSE)][:, GENE_IDX].mean(axis=0)).ravel() + 1e-9)
tert = pd.qcut(pd.Series(log_mean).rank(method='first'), 3, labels=False).to_numpy()
null_edges = []
for rep in range(5):
    perm = np.arange(len(GENE_IDX))
    for t in range(3):
        idx = np.flatnonzero(tert == t)
        perm[idx] = rng.permutation(idx)
    for i in range(len(GENE_IDX)):
        a = edges_by[MOUSE[0]].get(i, empty).reset_index(drop=True)
        b = edges_by[MOUSE[1]].get(int(perm[i]), empty).reset_index(drop=True)
        if len(a) and len(b):
            ag = ts.agreed_edges(a, b, max_distance=AGREE, window=WINDOW)
            if len(ag):
                null_edges.append(ag.assign(rep=rep))
NULL_EDGES = pd.concat(null_edges, ignore_index=True)
THRESHOLDS = (.25, .5, .75, 1., 1.5, 2.)
ph1, Q4P = [], {}
for seg in SEGMENTS:
    obs_in = GENE_EDGES[GENE_EDGES.location.eq(f'inside {seg}') & GENE_EDGES['class'].ne('mice disagree on width')]
    nul_in = NULL_EDGES[NULL_EDGES.location.eq(f'inside {seg}') & NULL_EDGES['class'].ne('mice disagree on width')]
    chosen = None
    for a_min in THRESHOLDS:
        o = obs_in[obs_in.amplitude.abs() >= a_min]
        n = nul_in[nul_in.amplitude.abs() >= a_min]
        ratio = (len(n) / 5) / max(len(o), 1)
        ph1.append({'segment': seg, '|amplitude| ≥': a_min, 'observed agreed interior edges': len(o),
                    'shuffled (mean of 5)': len(n) / 5, 'shuffled ÷ observed': ratio,
                    'observed sharp': int((o['class'] == 'sharp').sum()), 'shuffled sharp': (n['class'] == 'sharp').sum() / 5})
        if chosen is None and ratio <= .1 and len(o):
            chosen = a_min
    o = obs_in[obs_in.amplitude.abs() >= chosen] if chosen is not None else obs_in.iloc[:0]
    Q4P[seg] = {'A*': chosen, 'edges': len(o),
                'reference width': float(np.percentile(o.width, Q4_PERCENTILE)) if len(o) >= Q4_MIN else np.nan,
                'sharp edges': int((o['class'] == 'sharp').sum())}
PH1 = pd.DataFrame(ph1)
Q4P = pd.DataFrame(Q4P).T
display(PH1.round(3))
display(Q4P)

grid_pooled = sc.to_segment_units(GRID, POOLED)
ph3 = []
for spec in MOUSE:
    pos = POSITION[BINS[spec][0]]
    for x0 in STEP_AT:
        raw0 = float(from_units(x0, MOUSE_CUTS))
        curve = ts.spline_step_response(pos, raw0, KNOTS, GRID)
        ph3.append({'specimen': spec, 'step at (mouse units)': x0, 'step at (pooled units)': float(sc.to_segment_units(raw0, POOLED)),
                    'width, mouse units': ts.crossing_width(to_mouse(GRID), curve),
                    'width, pooled units': ts.crossing_width(grid_pooled, curve), 'overshoot': float(curve.max() - 1)})
PH3 = pd.DataFrame(ph3)
display(PH3.groupby('step at (mouse units)').mean(numeric_only=True).round(2))


def reference_post_hoc(location):
    if location in ('at S1/S2', 'at S2/S3'):
        return reference_at(location)
    return float(Q4P.loc[location.split()[-1], 'reference width']) if location else np.nan


def reading_post_hoc(row):
    if not isinstance(row.get('location'), str) or not row['location']:
        return row['class']
    if thin_at(row['position']) >= .25:
        return 'at the PT → thin-limb identity switch'
    if row['class'] in ('sharp', 'mice disagree on width'):
        return row['class']
    ref = reference_post_hoc(row['location'])
    if np.isfinite(ref) and ref <= SHARP:
        return row['class']
    return f"{row['class']}, or a blurred step (unresolved here)"


EDGES['thin-limb identity at edge'] = [thin_at(p) if np.isfinite(p) else np.nan for p in EDGES['position']]
EDGES['step reference (post hoc)'] = [reference_post_hoc(l) if isinstance(l, str) else np.nan for l in EDGES['location']]
EDGES['reading (post hoc)'] = [reading_post_hoc(r) for _, r in EDGES.iterrows()]
MAIN = EDGES[EDGES.main].copy()
display(pd.crosstab([EDGES['segment (Chen)'], EDGES['main']], EDGES['reading (post hoc)']))
display(EDGES[EDGES.representative & EDGES['class'].isin(['sharp', 'graded', 'gradual'])]
        [['segment (Chen)', 'name', 'main', 'mouse position', 'width A', 'width B', 'amplitude', 'reading (post hoc)']].round(2))

## 8 · Figures

- **85a:** left, Chen's PTS1/PTS2/PTS3 means of each representative's core (the segment-level view); right, the
  representative's binned profile (mean of the two mice, mouse units). Marks on agreed edges: filled sharp,
  half-filled graded, open gradual; grey when unresolved.
- **85b:** a sharp and a gradual pathway from the same Chen class; both mice's bins and fits; the label composition
  of each bin underneath.
- **85c:** resolution: Q1 spline step responses against the binned fit (Q2); Q3 label fractions; agreed gene edge
  widths by location.

In [ ]:
import textwrap

from matplotlib.colors import TwoSlopeNorm

raw = pd.read_csv(CHEN, sep='\t', compression='gzip')
pattern = re.compile(r'^m(\d+)_([A-Z0-9]+)_([a-c])$')
samples = pd.DataFrame([{'column': c, 'mouse': m.group(1), 'structure': m.group(2)}
                        for c in raw.columns if (m := pattern.match(str(c)))])
pt_cols = samples[samples.structure.isin(['PTS1', 'PTS2', 'PTS3'])]
tpm = raw.set_index('Gene_symbol')[samples.column.tolist()].astype(float)
tpm = tpm[tpm.index.notna()]
tpm = tpm.assign(_m=tpm[pt_cols.column].mean(axis=1).to_numpy()).sort_values('_m', ascending=False)
tpm = tpm[~tpm.index.duplicated(keep='first')].drop(columns='_m')
core_genes = sorted({g for c in CORES.values() for g in c})
log_pt = np.log2(tpm.loc[core_genes, pt_cols.column] + 1)
key = (pt_cols.structure + '|' + pt_cols.mouse).to_numpy()
mouse_level = log_pt.T.groupby(key).mean().T
groups = np.array([k.split('|')[0].replace('PT', '') for k in mouse_level.columns])


def chen_means(p):
    m = mouse_level.loc[CORES[p]].T.groupby(groups).mean().T
    return m.sub(m.mean(axis=1), axis=0).median().reindex(list(SEGMENTS)).to_numpy()


XG = np.linspace(.15, 2.95, 281)


def mean_profile(i):
    return np.mean([np.interp(XG, X[s], PW[s][i]) for s in MOUSE], axis=0)


CLASS_ORDER = {'sharp': 0, 'graded': 1, 'gradual': 2, 'mice disagree on width': 3, 'no agreed edge': 4, 'flat': 5}


def mark(ax, x, y, row, size=16):
    text = str(row.get('reading (post hoc)', ''))
    colour = GREY if ('unresolved' in text or 'thin-limb' in text) else INK
    fill = {'sharp': 'full', 'graded': 'left', 'gradual': 'none'}.get(row['class'], 'none')
    ax.plot([x], [y], marker='D' if 'thin-limb' in text else 'o', ms=size ** .5, fillstyle=fill, color=colour,
            markerfacecoloralt='white', mec=colour, mew=.6, zorder=3, ls='none')


def figure_a(stem):
    reps = MAIN[MAIN.representative].copy()
    reps['order'] = reps['class'].map(CLASS_ORDER)
    reps = reps.sort_values(['segment (Chen)', 'order', 'position'], na_position='last')
    order = reps.key.tolist()
    idx = [PATHWAYS.index(p) for p in order]
    chen = np.array([chen_means(p) for p in order])
    curves = np.array([mean_profile(i) for i in idx])
    height = (4.0 * len(order) + 22) * MM
    fig, axes = plt.subplots(2, 2, figsize=(170 * MM, height),
                             gridspec_kw={'width_ratios': [1, 4.2], 'wspace': .04, 'height_ratios': [len(order), 1.2], 'hspace': .02})
    (left, right), (blank, strip) = axes
    blank.axis('off')
    thin = np.mean([np.interp(XG, X[s], THIN[s]) for s in MOUSE], axis=0)
    strip.imshow(thin[None], cmap='Greys', vmin=0, vmax=1, aspect='auto', extent=(XG[0], XG[-1], 0, 1))
    strip.set_yticks([])
    for b in (1, 2):
        strip.axvline(b, color=INK, lw=.6)
    lim = max(np.nanmax(np.abs(chen)), .5)
    left.imshow(chen, cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim, lim), aspect='auto')
    left.set_xticks(range(3))
    left.set_xticklabels(['PTS1', 'PTS2', 'PTS3'])
    left.set_yticks(range(len(order)))
    left.set_yticklabels([short(p.split('|', 1)[1]) for p in order])
    lim2 = max(np.nanpercentile(np.abs(curves), 99), .25)
    right.imshow(curves, cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim2, lim2), aspect='auto',
                 extent=(XG[0], XG[-1], len(order) - .5, -.5))
    for b in (1, 2):
        right.axvline(b, color=INK, lw=.6)
    for r, p in enumerate(order):
        for _, e in EDGES[EDGES.key.eq(p) & EDGES['class'].isin(['sharp', 'graded', 'gradual'])].iterrows():
            mark(right, e['position'], r, e)
    breaks = np.flatnonzero(np.diff([SEGMENTS.index(s) for s in reps['segment (Chen)']])) + .5
    for ax in (left, right):
        for y in breaks:
            ax.axhline(y, color=INK, lw=.8)
    right.set_yticks([])
    right.set_xlim(XG[0], XG[-1])
    right.tick_params(labelbottom=False)
    strip.set_xlim(XG[0], XG[-1])
    strip.set_xlabel('elastic curve (mouse segment units)')
    save(fig, stem)


figure_a('fig85a_pathway_transitions')


def label_strip(ax, spec):
    rows, b = BINS[spec]
    seg = SEGMENT[rows]
    bottom = np.zeros(N_BINS)
    widths = np.gradient(X[spec])
    for s in SEGMENTS:
        f = np.bincount(b, weights=np.char.endswith(seg.astype(str), s).astype(float), minlength=N_BINS) / np.bincount(b, minlength=N_BINS)
        ax.bar(X[spec], f, width=widths, bottom=bottom, color=SEG_COLOR[s], lw=0)
        bottom += f
    ax.set_yticks([])
    ax.set_ylim(0, 1)
    for k in (1, 2):
        ax.axvline(k, color=INK, lw=.6)


def identity_strip(ax, spec):
    """Per bin: units of thin-limb primary identity (dark), PT identity (light), other (white)."""
    rows, b = BINS[spec]
    n = np.bincount(b, minlength=N_BINS)
    thin = THIN[spec]
    pt = np.bincount(b, weights=(IDENT[rows] == 'Proximal_Tubule').astype(float), minlength=N_BINS) / n
    widths = np.gradient(X[spec])
    ax.bar(X[spec], thin, width=widths, color='#4d4d4d', lw=0)
    ax.bar(X[spec], pt, width=widths, bottom=thin, color='#c7d9e8', lw=0)
    ax.set_yticks([])
    ax.set_ylim(0, 1)
    for k in (1, 2):
        ax.axvline(k, color=INK, lw=.6)


def example_panel(ax, strip, p):
    i = PATHWAYS.index(p)
    for spec in MOUSE:
        ax.plot(X[spec], PW[spec][i], 'o', ms=1.8, color=SPEC_COLOR[spec])
        m = PW_FITS[spec][0].loc[i]
        e = PW_FITS[spec][1].query('profile == @i')
        if len(e):
            params = [v for _, r in e.iterrows() for v in (r.position, r.width)]
            Xd = ts._design(X[spec], params)
            beta = np.linalg.lstsq(Xd, PW[spec][i], rcond=None)[0]
            xs = np.linspace(X[spec][0], X[spec][-1], 400)
            ax.plot(xs, ts._design(xs, params) @ beta, color=SPEC_COLOR[spec], lw=1)
    for _, e in EDGES[EDGES.key.eq(p) & EDGES['class'].isin(['sharp', 'graded', 'gradual'])].iterrows():
        mark(ax, e['position'], ax.get_ylim()[1], e, size=22)
    for k in (1, 2):
        ax.axvline(k, color=LIGHT, lw=.6)
    ax.axhline(0, color=LIGHT, lw=.5)
    ax.set_title(textwrap.fill(short(p.split('|', 1)[1]), 38), loc='left')
    ax.set_xlim(XG[0], XG[-1])
    ax.tick_params(labelbottom=False)
    label_strip(strip[0], MOUSE[0])
    strip[0].tick_params(labelbottom=False)
    identity_strip(strip[1], MOUSE[0])
    for st in strip:
        st.set_xlim(XG[0], XG[-1])
    strip[1].set_xlabel('elastic curve (mouse segment units)')


def pick_examples():
    reps = MAIN[MAIN.representative & MAIN['class'].isin(['sharp', 'graded', 'gradual'])]
    for seg in ('S2', 'S1', 'S3'):
        sub = reps[reps['segment (Chen)'].eq(seg)]
        sharp = sub[sub['class'].eq('sharp')]
        slow = sub[sub['class'].eq('gradual') & ~sub.reading.str.contains('unresolved')]
        if len(slow) == 0:
            slow = sub[sub['class'].eq('gradual')]
        if len(sharp) and len(slow):
            return [sharp.loc[sharp.amplitude.abs().idxmax(), 'key'], slow.loc[slow.amplitude.abs().idxmax(), 'key']]
    if len(reps) >= 2:
        return [reps.loc[reps.width.idxmin(), 'key'], reps.loc[reps.width.idxmax(), 'key']]
    return []



def examples_figure(keys, stem):
    fig = plt.figure(figsize=(62 * MM * len(keys), 70 * MM))
    gs = fig.add_gridspec(3, len(keys), height_ratios=[5, .8, .8], hspace=.1, wspace=.22)
    for c, p in enumerate(keys):
        ax = fig.add_subplot(gs[0, c])
        strips = [fig.add_subplot(gs[1, c], sharex=ax), fig.add_subplot(gs[2, c], sharex=ax)]
        example_panel(ax, strips, p)
        if c == 0:
            ax.set_ylabel('log2, centred')
    save(fig, stem)


EXAMPLES = pick_examples()
if EXAMPLES:
    examples_figure(EXAMPLES, 'fig85b_sharp_and_gradual')
else:
    print('No pair of classified representatives; 85b not drawn')

# Post hoc (85d): rule-picked examples. Sharp: the representative with the largest |amplitude| sharp edge that is not
# at the thin-limb switch. Gradual: the largest |amplitude| main edge whose post hoc reading is plain 'gradual'.
# Thin-limb: the S2 representative with the largest main amplitude that also has an edge at the thin-limb switch.
reps_e = EDGES[EDGES.representative & EDGES['class'].isin(['sharp', 'graded', 'gradual'])]
picks = []
sharp_e = reps_e[reps_e['reading (post hoc)'].eq('sharp')]
if len(sharp_e):
    picks.append(sharp_e.loc[sharp_e.amplitude.abs().idxmax(), 'key'])
grad_e = reps_e[reps_e.main & reps_e['reading (post hoc)'].eq('gradual')]
if len(grad_e):
    picks.append(grad_e.loc[grad_e.amplitude.abs().idxmax(), 'key'])
tl_keys = set(reps_e.loc[reps_e['reading (post hoc)'].str.contains('thin-limb'), 'key'])
tl_main = reps_e[reps_e.main & reps_e.key.isin(tl_keys) & reps_e['segment (Chen)'].eq('S2') & ~reps_e.key.isin(picks)]
if len(tl_main):
    picks.append(tl_main.loc[tl_main.amplitude.abs().idxmax(), 'key'])
EXAMPLES_D = picks
print('85d examples:', [short(k.split('|', 1)[1]) for k in EXAMPLES_D])
if EXAMPLES_D:
    examples_figure(EXAMPLES_D, 'fig85d_examples_post_hoc')


def figure_c(stem):
    fig, axes = plt.subplots(1, 3, figsize=(170 * MM, 48 * MM), gridspec_kw={'wspace': .35})
    ax = axes[0]
    spec = MOUSE[0]
    rows, b = BINS[spec]
    pos = POSITION[rows]
    gu = to_mouse(GRID)
    for x0 in STEP_AT:
        raw0 = float(from_units(x0, MOUSE_CUTS))
        ax.plot(gu, ts.spline_step_response(pos, raw0, KNOTS, GRID), color=INK, lw=.8)
        ax.plot([gu[0], x0, x0, gu[-1]], [0, 0, 1, 1], color=LIGHT, lw=.6, zorder=0)
    for kn in to_mouse(KNOTS):
        ax.axvline(kn, color=GREY, lw=.5, ls=':')
    ax.set_xlim(XG[0], XG[-1])
    ax.set_xlabel('mouse segment units')
    ax.set_ylabel('fitted step')
    ax = axes[1]
    for spec in MOUSE:
        for k in (1, 2):
            ax.plot(X[spec], LABEL_FRAC[(spec, k)], 'o', ms=1.6, color=SPEC_COLOR[spec])
            _, e = ts.fit_edges(X[spec], LABEL_FRAC[(spec, k)][None], flat=.1)
            params = [v for _, r in e.iterrows() for v in (r.position, r.width)]
            Xd = ts._design(X[spec], params)
            beta = np.linalg.lstsq(Xd, LABEL_FRAC[(spec, k)], rcond=None)[0]
            xs = np.linspace(X[spec][0], X[spec][-1], 400)
            ax.plot(xs, ts._design(xs, params) @ beta, color=SPEC_COLOR[spec], lw=.8)
    ax.set_xlim(XG[0], XG[-1])
    ax.set_xlabel('mouse segment units')
    ax.set_ylabel('later-segment label fraction')
    ax = axes[2]
    locs = ['inside S1', 'at S1/S2', 'inside S2', 'at S2/S3', 'inside S3']
    ok = GENE_EDGES[GENE_EDGES['class'].ne('mice disagree on width')]
    rng = np.random.default_rng(85)
    for k, loc in enumerate(locs):
        w = ok.loc[ok.location.eq(loc), 'width'].to_numpy()
        ax.scatter(k + rng.uniform(-.3, .3, len(w)), w, s=1.2, color=INK, alpha=.35, lw=0)
        if len(w):
            ax.plot([k - .35, k + .35], [np.percentile(w, Q4_PERCENTILE)] * 2, color=SEG_COLOR['S3'], lw=1)
    for y in (SHARP, GRADUAL):
        ax.axhline(y, color=GREY, lw=.6, ls='--')
    ax.set_yscale('log')
    ax.set_xticks(range(len(locs)))
    ax.set_xticklabels(locs, rotation=35, ha='right')
    ax.set_ylabel('agreed gene edge width')
    save(fig, stem)


figure_c('fig85c_resolution')

## 9 · Outputs and run record

In [ ]:
S0.to_csv(tables / 'positions_in_mouse_units.csv', index=False)
RESOLUTION.to_csv(tables / 'resolution.csv', index=False)
EDGES.to_csv(tables / 'pathway_edges.csv', index=False)
pd.concat({spec: PW_FITS[spec][1].assign(pathway=lambda d: [PATHWAYS[i] for i in d.profile]) for spec in MOUSE},
          names=['specimen']).to_csv(tables / 'pathway_edges_per_mouse.csv')
pd.concat({spec: PW_FITS[spec][0].assign(pathway=PATHWAYS) for spec in MOUSE}, names=['specimen']).to_csv(tables / 'pathway_models.csv')
GENE_EDGES.to_csv(tables / 'gene_edges_agreed.csv.gz', index=False)
PH1.to_csv(tables / 'post_hoc_gene_edge_chance_agreement.csv', index=False)
Q4P.to_csv(tables / 'post_hoc_interior_reference.csv')
PH3.to_csv(tables / 'post_hoc_spline_step_pooled_units.csv', index=False)
identity.to_csv(tables / 'post_hoc_unit_identity_by_bin.csv', index=False)
np.savez_compressed(tables / 'profiles.npz', pathways=np.array(PATHWAYS), **{f'x|{s}': X[s] for s in MOUSE},
                    **{f'pathway|{s}': PW[s] for s in MOUSE}, **{f'reference|{s}': REF[s] for s in MOUSE})
READING.extend([
    f"Logic version {NOTEBOOK_LOGIC_VERSION!r}, as the protocol names it.",
    'P1: a bin\'s position is the median curve position of its units, converted to mouse units afterwards.',
    'Q1: the spline is fitted by least squares to the 0/1 step on the mouse\'s own units with 71\'s stored knots and '
    'evaluated on 71\'s grid; its width is the distance between the first 0.1 and 0.9 crossings.',
    'Q2, Q3: label and step fractions are fitted with E1 at a flat threshold of 0.1 (a fraction, not log2); the edge '
    'nearest the boundary (rising, for Q3) gives the width.',
    'Q3 at a boundary uses the mean of the two mice\'s label widths.',
    'R3: a pathway with no agreed edge is "flat" only when both mice choose no edge; otherwise "no agreed edge".',
    'G2 runs on every agreed edge of a pathway; the figure marks every agreed edge, the tables flag the main one.',
    'Post hoc PH1: Q4 reached the fit\'s lower bound in every segment because chance-agreed noise edges are sharp; Q4′ '
    '(amplitude-filtered against a shuffled pairing) replaces it in "reading (post hoc)". The frozen "reading" is kept.',
    'Post hoc PH2: edges where ≥ 25% of units have thin-limb primary identity are read as the PT → thin-limb identity '
    'switch. The figures mark them with diamonds and draw unresolved and identity-switch edges in grey.',
    'Post hoc PH3: the spline step response in pooled units, for comparison with 81.',
    'Post hoc 85d: three rule-picked examples (sharp inside the PT, resolved gradual, an S2 pathway at the thin-limb '
    'switch). 85a and 85b draw identity strips.',
])
pd.Series(READING, name='how the protocol was read').to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
          'protocol': 'docs/results/pt-transition-sharpness.md (1971e97, amended 72ad1fb)',
          'pass1_fingerprint': FINGERPRINT, 'chen_sha256': CHEN_SHA, 'guards': GUARDS,
          'mouse_cuts': MOUSE_CUTS, 'pooled_cuts': POOLED,
          'resolution': RESOLUTION.to_dict('records'), 'main_edges': summary.to_dict('index'),
          'gene_edges': {'genes fitted': int(len(GENE_IDX)), 'genes with an agreed edge': int(GENE_EDGES.gene.nunique()),
                         'agreed edges': int(len(GENE_EDGES))},
          'post_hoc': {'interior_reference': Q4P.to_dict('index'),
                       'readings': EDGES['reading (post hoc)'].value_counts().to_dict(),
                       'main_readings': pd.crosstab(MAIN['segment (Chen)'], MAIN['reading (post hoc)']).to_dict('index'),
                       'examples_85d': EXAMPLES_D},
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
for line in READING:
    print('·', line)
print(f"Run record written · {record['seconds']} s")